# Tổng hợp các thí nghiệm HeteroES

Bảng gom kết quả các notebook thí nghiệm (`results/*.json`). "Đúng" = điểm 1 trên tập test, chấm **một lần** ở cuối;
p là kiểm định McNemar chính xác (mô hình gốc so với checkpoint được chọn). Cột "đúng / đã xong" là tỉ lệ đúng trên các
câu mô hình trả lời xong trong giới hạn token: nếu cột này không tăng thì mức tăng chỉ là do viết gọn hơn.

In [ ]:
import json
from pathlib import Path

HERE = Path.cwd()


def table(folder):
    print(f"{'thí nghiệm':22} {'mô hình':30} {'n':>5} {'gốc':>5} {'sau ES':>7} {'p':>9}  {'đúng/đã xong: gốc → sau':>24}")
    for path in sorted((folder / "results").glob("*.json")):
        s = json.loads(path.read_text())
        t = s["test"]
        f = s.get("finished") or {}
        cells = [v for k, v in f.items()]
        ratio = (lambda x: x["rightAmongFinished"] / max(x["finished"], 1))
        change = f"{ratio(cells[0]):6.1%} → {ratio(cells[-1]):6.1%}" if cells else "—"
        print(f"{s['name']:22} {s['model'][:30]:30} {t['base']['count']:5} {t['base']['correct']:5} "
              f"{t['selected']['correct']:7} {s['comparison']['mcnemarExactP']:9.2g}  {change:>24}")


table(HERE)

## Tốc độ: 1 GPU so với nhiều GPU

Trung vị số giây một thế hệ, đọc từ ledger của coordinator. Mốc 1 GPU là một run ngắn 6 thế hệ chỉ trên 5070 Ti của
Nhân, cùng cấu hình.

In [ ]:
for path in sorted((HERE / "results").glob("*.json")):
    s = json.loads(path.read_text())
    speed = s.get("speed") or {}
    one = speed.get("5070ti-nhan", {}).get("medianSeconds")
    print(s["name"])
    for machines, v in sorted(speed.items(), key=lambda kv: kv[0].count("+")):
        gain = f"  nhanh gấp {one / v['medianSeconds']:.2f}" if one and machines != "5070ti-nhan" else ""
        print(f"   {machines:50} {v['generations']:4} thế hệ  {v['medianSeconds']:6.1f} s/thế hệ{gain}")

## Lần 1 (đêm 24 → 25/11) và các run trước đó

Lần 1 nằm trong `v1/` (notebook, kết quả, quyết định). Phát hiện chính, và là lý do làm lại:

| Kết quả lần 1 | Có ý nghĩa? | Vì sao |
|---|---|---|
| E1 GSM8K × Qwen 0.5B, 462 → 528/1319 | không | 320 token: tỉ lệ đúng trên câu đã xong **giảm** 51% → 47%; mức tăng chỉ do trả lời xong nhiều câu hơn |
| E3 GSM8K × Qwen 1.5B, 703 → 871/1319 | không | 320 token: đúng/đã xong 77,5% → 76,8%; 170 trong 251 câu được sửa là câu mà mô hình gốc bị cắt |
| E2 phép nhân × SmolLM2-360M, 0 → 0 | không | mô hình không học được gì |
| E4 Countdown × Qwen 1.5B, 4 → 31/250 | **có** | đúng/đã xong 2,1% → 12,4%; mô hình gốc chỉ 1,6% kể cả với 1024 token |
| Countdown × Qwen 0.5B (24/11, ngoài lab), 0 → 33/250 | **có** | mô hình gốc 0/250 kể cả với 1024 token |
| Phép nhân × Qwen 0.5B (trước 24/11, ngoài lab), 95 → 186/300 | một phần | phần lớn là học ghi `\boxed{}` (179 → 300 câu có ô), nhưng đúng trên câu có ô vẫn tăng 53% → 62% |

Lần 1 cũng gần như không phân tán: E1 và E2 chạy hoàn toàn trên một 5070 Ti, E3 chỉ có máy Quang góp 14 trong 3200
candidate; chỉ E4 chạy trên 2 GPU (190/190 audit khớp).

In [ ]:
table(HERE / "v1")